<a href="https://colab.research.google.com/github/Higashi88jp/Matsuo_Lab/blob/main/DL_Basic_2026_Spring_Competition_EEG_baselin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning 基礎講座　最終課題: 脳波分類

## 概要
被験者が画像を見ているときの脳波から，その画像がどのカテゴリに属するかを分類するタスク．
- サンプル数: 訓練 118,800 サンプル，検証 59,400 サンプル，テスト 59,400 サンプル
- クラス数: 5
- 入力: 脳波データ（チャンネル数 x 系列長）
- 出力: 対応する画像のクラス
- 評価指標: Top-1 accuracy

### 元データセット ([Gifford2022 EEG dataset](https://osf.io/3jk45/)) との違い

- 本コンペでは難易度調整の目的で元データセットにいくつかの改変を加えています．

1. 訓練セットのみの使用
  - 元データセットでは訓練データに存在しなかったクラスの画像を見ているときの脳波においてテストが行われますが，これは難易度が非常に高くなります．
  - 本コンペでは元データセットの訓練セットを再分割し，訓練時に存在した画像に対応する別の脳波において検証・テストを行います．

2. クラス数の減少
  - 元データセット（の訓練セット）では16,540枚の画像に対し，1,654のクラスが存在します．
    - e.g. `aardvark`, `alligator`, `almond`, ...
  - 本コンペでは1,654のクラスを，`animal`, `food`, `clothing`, `tool`, `vehicle`の5つにまとめています．
    - e.g. `aardvark -> animal`, `alligator -> animal`, `almond -> food`, ...

### 考えられる工夫の例

- 音声モデルの導入
  - 脳波と同じ波である音声を扱うアーキテクチャを用いることが有効であると知られています．
  - 例）Conformer [[Gulati+ 2020](https://arxiv.org/abs/2005.08100)]
- 画像データを用いた事前学習
  - 本コンペのタスクは脳波のクラス分類ですが，配布してある画像データを脳波エンコーダの事前学習に用いることを許可します．
  - 例）CLIP [Radford+ 2021]
  - 画像を用いる場合は[こちら](https://osf.io/download/3v527/)からダウンロードしてください．
- 過学習を防ぐ正則化やドロップアウト


## 修了要件を満たす条件
- ベースラインモデルのbest test accuracyは38.8%となります．**これを超えた提出のみ，修了要件として認めます**．
- ベースラインから改善を加えることで，55%までは性能向上することを運営で確認しています．こちらを 1 つの指標として取り組んでみてください．

## 注意点
- 最終的な予測モデルは，**配布している訓練データを用いて学習**（ファインチューニング含む）したものとしてください．
- 学習を行わず，**事前学習済みモデルの知識のみを利用した推論は禁止**します．  
（例: ChatGPT 等の LLM に入力して推論を得るのみ）

### 事前学習モデルの利用
許可される事項
- **構成要素としての事前学習モデルの利用**: 自身で実装したアーキテクチャの一部（特徴抽出，埋め込みなど）として事前学習モデル（BERT，ViT など）を利用することは可能です．
- **ファインチューニング**: 上記の用途で利用している事前学習モデルのファインチューニングは可能です．

禁止される事項  
- **タスク解決用の事前学習モデルの利用**: transformers などで提供されている，対象タスクを直接解くための事前学習モデルでそのまま推論のみ，またはファインチューニングのみで利用することは禁止とします．
  - 禁止事項の例: VQA タスクを直接解くための事前学習モデルを VQA タスクで利用する．

## 1.準備

In [1]:
# omnicampus 実行用
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 29.2 MB/s eta 0:00:00


In [2]:
# ライブラリのインポートとシード固定
import os, sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from einops.layers.torch import Rearrange
from einops import repeat
from glob import glob
from termcolor import cprint
from tqdm.notebook import tqdm
from torch.amp import autocast

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
# ドライブのマウント（Colabの場合）
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ワーキングディレクトリを作成し移動．ノートブックを配置したディレクトリに適宜書き換え
# WORK_DIR = "/content/drive/MyDrive/weblab/DLBasics2025/Competition"
WORK_DIR = "/content/drive/MyDrive/Colab Notebooks/DLBasics2026/Competition/EEG"
os.makedirs(WORK_DIR, exist_ok=True)
%cd {WORK_DIR}

/content/drive/MyDrive/Colab Notebooks/DLBasics2026/Competition/EEG


## 2.データセット

ノートブックと同じディレクトリに`data/`が存在することを確認してください．

In [5]:
class ThingsEEGDataset(torch.utils.data.Dataset):
    def __init__(self, split: str, stats=None) -> None:
        super().__init__()

        assert split in ["train", "val", "test"]
        self.split = split
        self.num_classes = 5
        self.num_subjects = 10

        self.X = torch.from_numpy(np.load(f"data/{split}/eeg.npy")).float()
        self.subject_idxs = torch.from_numpy(np.load(f"data/{split}/subject_idxs.npy")).long()

        self.subject_idxs = self.subject_idxs - 1
        self.num_subjects = int(self.subject_idxs.max().item()) + 1

        if split in ["train", "val"]:
            self.y = torch.from_numpy(np.load(f"data/{split}/labels.npy")).long()

        if stats is None and split == "train":
            # (N, C, T) → channelごとに平均
            self.mean = self.X.mean(dim=(0, 2), keepdim=True)
            self.std = self.X.std(dim=(0, 2), keepdim=True) + 1e-6
        else:
            self.mean = stats["mean"]
            self.std = stats["std"]

        print(f"EEG: {self.X.shape}")

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        X = (self.X[i] - self.mean.squeeze(0)) / self.std.squeeze(0)

        if self.split == "train":
            noise_std = torch.empty(1).uniform_(0.0, 0.02).item()
            X = X + noise_std * torch.randn_like(X)

        if hasattr(self, "y"):
            return X, self.y[i], self.subject_idxs[i]
        else:
            return X, self.subject_idxs[i]

    def get_stats(self):
        return {"mean": self.mean, "std": self.std}

    @property
    def num_channels(self):
        return self.X.shape[1]

    @property
    def seq_len(self):
        return self.X.shape[2]


## 3.ベースラインモデル

In [6]:
class ConvBlock(nn.Module):
    def __init__(
        self,
        in_dim,
        out_dim,
        kernel_size: int = 15,  # 7 → 11 or 15
        p_drop: float = 0.2,
    ) -> None:
        super().__init__()

        self.conv0 = nn.Conv1d(in_dim, out_dim, kernel_size, padding="same")
        self.conv1 = nn.Conv1d(out_dim, out_dim, kernel_size, padding="same")

        # GroupNorm
        self.norm0 = nn.GroupNorm(8, out_dim)
        self.norm1 = nn.GroupNorm(8, out_dim)

        # skip connection 用の projection
        self.proj = None
        if in_dim != out_dim:
            self.proj = nn.Conv1d(in_dim, out_dim, kernel_size=1)

        self.dropout = nn.Dropout(p_drop)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        residual = X if self.proj is None else self.proj(X)

        X = self.conv0(X)
        X = F.gelu(self.norm0(X))
        X = X + residual

        residual = X
        X = self.conv1(X)
        X = F.gelu(self.norm1(X))
        X = X + residual

        return self.dropout(X)

class ConvTransformerClassifier(nn.Module):
    def __init__(
        self,
        num_classes,
        seq_len,
        in_channels,
        num_subjects,
        hid_dim=128,
        n_heads=4,
        n_layers=2,
        p_drop=0.2,
    ):
        super().__init__()

        # CNN部分（特徴抽出）
        self.stem = nn.Sequential(
            ConvBlock(in_channels, hid_dim, kernel_size=15, p_drop=p_drop),
            ConvBlock(hid_dim, hid_dim, kernel_size=15, p_drop=p_drop),
        )

        # 時間を半分に圧縮
        self.downsample = nn.AvgPool1d(2)
        reduced_len = seq_len // 2

        # position embedding
        self.pos_emb = nn.Parameter(torch.randn(1, reduced_len, hid_dim) * 0.02)

        # Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hid_dim,
            nhead=n_heads,
            dim_feedforward=hid_dim * 4,
            dropout=p_drop,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)

        # subject embedding
        self.subject_emb = nn.Embedding(num_subjects, 64)

        # classifier
        self.head = nn.Sequential(
            nn.LayerNorm(hid_dim + 64),
            nn.Linear(hid_dim + 64, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, X, subject_idxs):
        # (B, C, T)
        X = self.stem(X)        # (B, hid, T)
        X = self.downsample(X)  # (B, hid, T/2)
        X = X.transpose(1, 2)   # (B, T/2, hid)
        X = X + self.pos_emb[:, :X.size(1)]
        X = self.transformer(X) # (B, T/2, hid)
        X = X.mean(dim=1)       # (B, hid)
        s = self.subject_emb(subject_idxs)
        s = F.layer_norm(s, s.shape[1:])
        s = F.dropout(s, p=0.2, training=self.training)
        X = torch.cat([X, s], dim=1)
        return self.head(X)

class BasicConvClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int,
        seq_len: int,
        in_channels: int,
        num_subjects: int,
        hid_dim: int = 128
    ) -> None:
        super().__init__()

        self.blocks = nn.Sequential(
            ConvBlock(in_channels, hid_dim),
            ConvBlock(hid_dim, hid_dim),
            ConvBlock(hid_dim, hid_dim),
            ConvBlock(hid_dim, hid_dim),
        )

        self.pool = nn.AdaptiveAvgPool1d(8)
        self.subject_emb = nn.Embedding(num_subjects, 64)
        self.head = nn.Sequential(
            nn.Linear(hid_dim * 8 + 64, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, X: torch.Tensor, subject_idxs: torch.Tensor) -> torch.Tensor:
        X = self.blocks(X)              # (B, hid_dim, T)
        X = self.pool(X).flatten(1)     # (B, hid_dim)

        s = self.subject_emb(subject_idxs)  # (B, 64)
        X = torch.cat([X, s], dim=1)        # (B, hid_dim+32)

        return self.head(X)


In [7]:
# for debag
train_set = ThingsEEGDataset("train")
stats = train_set.get_stats()
val_set = ThingsEEGDataset("val", stats=stats)
test_set = ThingsEEGDataset("test", stats=stats)

EEG: torch.Size([118800, 17, 100])
EEG: torch.Size([59400, 17, 100])
EEG: torch.Size([59400, 17, 100])


In [8]:
# for debag
print("train subject min/max:", train_set.subject_idxs.min().item(), train_set.subject_idxs.max().item())
print("val   subject min/max:", val_set.subject_idxs.min().item(), val_set.subject_idxs.max().item())
print("test  subject min/max:", test_set.subject_idxs.min().item(), test_set.subject_idxs.max().item())
print("train unique:", torch.unique(train_set.subject_idxs))


train subject min/max: 0 9
val   subject min/max: 0 9
test  subject min/max: 0 9
train unique: tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


## 4.訓練実行

In [9]:
# ハイパラ
lr = 1e-4          # 0602 学習率調整 lr = 0.001 → 0.0003 か 0.0001
batch_size = 256
epochs = 80

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# ------------------
#    Dataloader
# ------------------
train_set = ThingsEEGDataset("train")

print("subject min/max:", train_set.subject_idxs.min().item(), train_set.subject_idxs.max().item())

train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

stats = train_set.get_stats()

val_set = ThingsEEGDataset("val", stats=stats)

val_loader = torch.utils.data.DataLoader(
    val_set,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

test_set = ThingsEEGDataset("test", stats=stats)

# ------------------
#       Model
# ------------------
model = ConvTransformerClassifier(
    num_classes=train_set.num_classes,
    seq_len=train_set.seq_len,
    in_channels=train_set.num_channels,
    num_subjects=train_set.num_subjects
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs,
    eta_min=1e-6
)

scaler = torch.amp.GradScaler("cuda")

# ------------------
#   Start training
# ------------------
max_val_acc = 0
def accuracy(y_pred, y):
    return (y_pred.argmax(dim=-1) == y).float().mean()

writer = SummaryWriter("tensorboard")

max_val_acc = 0
patience = 8
counter = 0
best_epoch = 0

for epoch in range(epochs):
    model.train()
    train_loss, train_acc = [], []

    for X, y, subject_idxs in train_loader:
        X = X.to(device)
        y = y.to(device).long()
        subject_idxs = subject_idxs.to(device)

        if random.random() < 0.5:
            X = X + 0.01 * torch.randn_like(X)

        if random.random() < 0.5:
            shift = random.randint(-20, 20)
            X = torch.roll(X, shifts=shift, dims=-1)

        lam = np.random.beta(0.4, 0.4)
        index = torch.randperm(X.size(0)).to(device)

        X_mix = lam * X + (1 - lam) * X[index]
        y_a, y_b = y, y[index]

        optimizer.zero_grad()

        with autocast():
            y_pred = model(X_mix, subject_idxs)
            loss = lam * F.cross_entropy(y_pred, y_a, label_smoothing=0.05) + \
            (1 - lam) * F.cross_entropy(y_pred, y_b, label_smoothing=0.05)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        train_loss.append(loss.item())
        train_acc.append((y_pred.argmax(dim=-1) == y_a).float().mean().item())

    model.eval()
    val_loss, val_acc = [], []
    with torch.no_grad():
        for X, y, subject_idxs in val_loader:
            X = X.to(device)
            y = y.to(device).long()
            subject_idxs = subject_idxs.to(device)

            with autocast():
                y_pred = model(X, subject_idxs)
                loss = F.cross_entropy(y_pred, y)

            val_loss.append(loss.item())
            val_acc.append((y_pred.argmax(dim=-1) == y).float().mean().item())

    scheduler.step()

    print(f"Epoch {epoch+1}/{epochs} | \
        train loss: {np.mean(train_loss):.3f} | \
        train acc: {np.mean(train_acc):.3f} | \
        val loss: {np.mean(val_loss):.3f} | \
        val acc: {np.mean(val_acc):.3f}")

    val_mean = np.mean(val_acc)

    print(f"Epoch {epoch+1} | val acc: {val_mean:.4f}")

    if val_mean > max_val_acc:
        print("New best! Saving model.")
        max_val_acc = val_mean
        best_epoch = epoch
        counter = 0

        torch.save(model.state_dict(), "model_best.pt")
    else:
        counter += 1
        print(f"No improvement: {counter}/{patience}")

    if counter >= patience:
        print("Early stopping triggered!")
        break

    writer.add_scalar("train_loss", np.mean(train_loss), epoch)
    writer.add_scalar("train_acc", np.mean(train_acc), epoch)
    writer.add_scalar("val_loss", np.mean(val_loss), epoch)
    writer.add_scalar("val_acc", np.mean(val_acc), epoch)

    torch.save(model.state_dict(), "model_last.pt")


device: cpu
EEG: torch.Size([118800, 17, 100])
subject min/max: 0 9
EEG: torch.Size([59400, 17, 100])
EEG: torch.Size([59400, 17, 100])


/tmp/ipykernel_11314/2335426759.py:76: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)
/tmp/ipykernel_11314/2439982406.py:62: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = torch.amp.GradScaler("cuda")
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TypeError: autocast.__init__() missing 1 required positional argument: 'device_type'

In [ ]:
print(f"Best validation accuracy: {max_val_acc:.4f}")
print(f"Best epoch: {best_epoch + 1}")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir tensorboard

## 5.評価

In [ ]:
# ------------------
#    Dataloader
# ------------------
test_loader = torch.utils.data.DataLoader(
    test_set,
    batch_size=batch_size,
    shuffle=False
)

# ------------------
#       Model
# ------------------
model = ConvTransformerClassifier(
    num_classes=test_set.num_classes,
    seq_len=test_set.seq_len,
    in_channels=test_set.num_channels,
    num_subjects=test_set.num_subjects
).to(device)

model.load_state_dict(torch.load("model_best.pt", map_location=device))

# ------------------
#  Start evaluation
# ------------------
preds = []
model.eval()

for X, subject_idxs in tqdm(test_loader, desc="Evaluation"):
    X = X.to(device)
    subject_idxs = subject_idxs.to(device)

    preds.append(model(X, subject_idxs).detach().cpu())

preds = torch.cat(preds, dim=0).numpy()
np.save("submission.npy", preds)
print(f"Submission {preds.shape} saved.")


## 提出方法

以下の3点をzip化し，Omnicampusの「最終課題 (EEG)」から提出してください．

- `submission.npy`
- `model_last.pt`や`model_best.pt`など，テストに使用した重み（拡張子は`.pt`のみ）
- 本Colab Notebook

In [ ]:
print(preds.shape)


In [ ]:
# @title デフォルトのタイトル テキスト
from zipfile import ZipFile

model_path = "model_best.pt"
notebook_path = "/content/drive/MyDrive/Colab Notebooks/DLBasics2026/Competition/EEG/DL_Basic_2026_Spring_Competition_EEG_baseline.ipynb"

with ZipFile("submission.zip", "w") as zf:
    zf.write("submission.npy")
    zf.write(model_path)
    zf.write(notebook_path)


In [ ]:
from zipfile import ZipFile
from google.colab import drive

model_path = "model_best.pt"
notebook_path = "/content/drive/MyDrive/Colab Notebooks/DLBasics2026/Competition/EEG/DL_Basic_2026_Spring_Competition_EEG_baseline.ipynb"

with ZipFile("submission.zip", "w") as zf:
    zf.write("submission.npy")
    zf.write(model_path)
    zf.write(notebook_path, arcname="baseline.ipynb")


In [ ]:
!ls "/content/drive/MyDrive/Colab Notebooks/DLBasics2026/Competition/EEG"

In [ ]:
from google.colab import files
files.download('submission.zip')
